In [0]:
CREATE EXTENSION IF NOT EXISTS vector;

DROP TABLE IF EXISTS weather_documents CASCADE;
DROP TABLE IF EXISTS weather_embeddings CASCADE;

CREATE TABLE IF NOT EXISTS weather_documents (
id TEXT PRIMARY KEY,
location TEXT NOT NULL,
source_type TEXT NOT NULL,
headline TEXT,
narrative_text TEXT NOT NULL,
issued_at TIMESTAMPTZ,
payload JSONB,
synced_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE weather_embeddings (
id SERIAL PRIMARY KEY,
document_id TEXT NOT NULL,
chunk_index INTEGER NOT NULL,
chunk_text TEXT NOT NULL,
embedding VECTOR(384) NOT NULL,
model_name TEXT NOT NULL,
created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
CONSTRAINT fk_weather_document
FOREIGN KEY (document_id)
REFERENCES weather_documents(id)
ON DELETE CASCADE,
CONSTRAINT unique_weather_embedding_chunk 
  UNIQUE(document_id, chunk_index)
);

CREATE INDEX IF NOT EXISTS weather_embeddings_embedding_idx
ON weather_embeddings
USING hnsw (embedding vector_cosine_ops);